# Load required packages

In [ ]:
from enum import unique

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
#import pooch
from scipy.sparse import csr_matrix
from scipy.io import mmwrite
from pathlib import Path
import anndata
import copy
from scipy.io import mmread
from scipy.sparse import csr_matrix
import glob
import re
print("scanpy version:", sc.__version__)
print("numpy version:", np.__version__)

In [ ]:
os.getcwd()

In [ ]:
os.chdir('/mnt/d/lxk/project/jiangshucai20260506')
from helper.annotation_helper import *

In [ ]:
sc.settings.set_figure_params(dpi=300, dpi_save=500)
setGrid(True)

# Load and prepare data

In [ ]:
sampleInfo = [
    "LT-a",
    "LT-b",
    "LT-c",
    "LT-d",
    "LT-e",
    "LT-f",
    "LZ-c",
    "LZ-d",
    "LZ-f"
]
sampleList = list()
print(Path.cwd())
for sample in sampleInfo:

    obsfile = glob.glob(f".\\data\\scRNA-seq\\{sample}\\filter_matrix\\" + "\\*barcodes.tsv.gz")[0]
    varfile = glob.glob(f".\\data\\scRNA-seq\\{sample}\\filter_matrix\\" + "\\*features.tsv.gz")[0]
    mtxfile = glob.glob(f".\\data\\scRNA-seq\\{sample}\\filter_matrix\\" + "\\*matrix.mtx.gz")[0]
    print(obsfile + " "+ varfile+ " " + mtxfile)

    obs = pd.read_csv(obsfile,header=None,index_col=0,sep = '\t')
    var = pd.read_csv(varfile,header=None,index_col=0,sep = '\t')


    mtx = mmread(mtxfile)
    mtx = mtx.T
    mtx = csr_matrix(mtx)

    #修改标签
    var.index.name = None
    obs.index.name = None
    obs["sample"]  = sample

    if 'LT' in sample:
        group = 'LT'
    else:
        group = 'LZ'

    obs['group']  = group
    adata = sc.AnnData(mtx,obs = obs,var = var)
    adata.var_names_make_unique()
    sampleList.append(copy.deepcopy(adata))
    del adata

#adatas = anndata.concat(sampleList,join = 'inner')
adatas = anndata.concat(sampleList, join='outer', fill_value=0)
print(adatas.shape)
adatas.obs_names_make_unique()
print(adatas.shape)
adatas.obs['sample'].value_counts()


In [ ]:
sc.pp.filter_genes(adatas,min_cells=3)
sc.pp.filter_cells(adatas,min_genes=200)
adatas.var['mt'] = adatas.var_names.str.startswith('MT-')
adatas.var["ribo"] = adatas.var_names.str.startswith(("RPS","RPL"))# hemoglobin genes
adatas.var["hb"] = adatas.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(adatas,qc_vars=['mt','ribo','hb'],percent_top=None,log1p=False,inplace=True)
sc.pl.violin(adatas, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt','pct_counts_hb','pct_counts_ribo'],
             jitter=0.4, multi_panel=True)
sc.pp.scrublet(adatas, batch_key="sample") #pip install scikit-image

In [ ]:

#过滤
adatas = adatas[adatas.obs.n_genes_by_counts < 7000, :]
adatas = adatas[adatas.obs.n_genes_by_counts > 300, :]

adatas = adatas[adatas.obs.total_counts < 36000, :]
adatas = adatas[adatas.obs.predicted_doublet == False, :]

adatas = adatas[adatas.obs.pct_counts_mt < 10, :]
#adatas = adatas[adatas.obs.pct_counts_hb < 2, :]
#adatas = adatas[adatas.obs.pct_counts_ribo < 40, :]
sc.pl.violin(adatas, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)


adatas.layers['counts'] = adatas.X.copy()


In [ ]:
adatas.X = adatas.layers["counts"].copy()
sc.experimental.pp.highly_variable_genes(
    adatas,
    flavor='pearson_residuals',
    n_top_genes=3000,
    layer='counts',
    batch_key='sample'
)
sc.pp.normalize_total(adatas,target_sum=1e4)
sc.pp.log1p(adatas)

adatas.layers['normalize'] = adatas.X.copy()

# SCVI

In [ ]:
import scvi  # 变分自编码器框架（scVI/scANVI 模型）
import torch  # 底层深度学习后端 scvi需要用到GPU
print("device_count:", torch.cuda.device_count())
adata_hvg = adatas[:, adatas.var.highly_variable].copy()
scvi.model.SCVI.setup_anndata(adata_hvg, layer="counts", batch_key="sample")
model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30, gene_likelihood="nb")
model.train()
SCVI_LATENT_KEY = "X_scVI"
adatas.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

sc.pp.neighbors(adatas, use_rep=SCVI_LATENT_KEY,n_neighbors=15)
sc.tl.umap(adatas)

# leiden

In [ ]:
import importlib
import helper.annotation_helper as ah
import helper.plot_helper as ph
importlib.reload(ah)
importlib.reload(ph)


In [ ]:
for resolution in [0.3,0.5,0.7,0.9,1.1]:
    sc.tl.leiden(adatas,
                 key_added='clusters',
                 flavor="igraph",
                 n_iterations=-1,
                 resolution = resolution,
                 )
    sc.pl.umap(
            adatas,
            color='clusters',
            legend_loc = 'on data'
        )

In [ ]:
sc.tl.leiden(adatas,
                key_added='clusters',
                flavor="igraph",
                n_iterations=-1,#-1为直到收敛为最佳聚类，-2是默认值，迭代两次
                resolution = 0.5,#默认1，越高分的越多
                )
sc.pl.umap(
        adatas,
        color='clusters',
        legend_loc = 'on data'
    )

# annotation

In [ ]:
# 找到marker基因
sc.tl.rank_genes_groups(adatas, groupby="clusters",pts=True)
from scanpy.get import rank_genes_groups_df
df = rank_genes_groups_df(adatas, None)  # None 表示导出所有组

df = df[df['pct_nz_group'] > 0.8]
df = df[df['logfoldchanges'] > 1]

df = df[['group', 'names','logfoldchanges',"pct_nz_group"]]
df.to_csv(".\\export\\all_Marker_1.csv", index=False)

In [ ]:
marker_genes_dict = {
    "SFT_Tumor": ["IGF2", "LHX2", "NPW", "MGP", "ALDH1A1", "GRIA2", "PTGDS", "STAT6"],
    "Pericyte/SMC": ["RGS5", "NOTCH3", "PDGFRB", "MCAM", "NDUFA4L2", "ACTA2", "TAGLN"],
    "T/NK": ["CD3D", "CD3E", "TRAC", "TRBC2", "CCL5", "NKG7", "GNLY"],
    "Myeloid": ["PTPRC", "LYZ", "TYROBP", "AIF1", "LST1", "FCER1G", "CSF1R"],
    "Endo": ["PECAM1", "VWF", "FLT1", "PLVAP", "KDR", "RAMP2", "ENG"],
    "Mast": ["TPSAB1", "TPSB2", "CPA3", "KIT", "MS4A2"],
    "CAFs": ["LUM", "DCN", "COL1A1", "COL3A1", "SFRP2", "POSTN", "PDGFRA", "FAP", "COL6A1"],
    "Oligo": ["PLP1", "MAG", "MOBP", "MBP", "MOG"],
    "OPCs": ["PTPRZ1", "SOX10", "BCAN", "OLIG1", "OLIG2", "PDGFRA"]
}

sc.pl.dotplot(
    adatas,
    marker_genes_dict,
    groupby="celltype",
    dendrogram=False,
    cmap="YlGnBu", # https://matplotlib.org/stable/users/explain/colors/colormaps.html
    #dot_max=0.6,     # 点最大直径比例（默认 1.0）
    #dot_min=0.1,     # 点最小直径比例（默认 0）
    vmax=5,          # 显示的最大表达值，>3 的都显示为同一深红
    vmin=0,           # 最小值（默认0）
    #save="_markers.pdf"
    standard_scale="var"


)

In [ ]:
cluster2annotation = {
    "0": "SFT_Tumor",
    "1": "SFT_Tumor",
    "2": "SFT_Tumor",
    "3": "SFT_Tumor",
    "4": "SFT_Tumor",
    "5,0": "Myeloid",
    "5,1": "Myeloid",
    "5,2": "Myeloid",
    "5,3": "Myeloid",
    "5,4": "Myeloid",
    "5,5": "Myeloid",
    "5,6": "Myeloid",
    "5,7": "Myeloid",
    "6": "Pericyte/SMC",
    "7": "SFT_Tumor",
    "8": "T/NK",
    "9": "Myeloid",
    "10": "Myeloid",
    "11": "Endo",
    "12": "Myeloid",
    "13": "Mast",
    "16": "CAFs",
    "17": "SFT_Tumor",
    "18": "Oligo",
    "19": "OPCs"
}

In [ ]:
sc.pl.umap(
        adatas,
        color='group',
        #legend_loc = 'on data'
    )

In [ ]:
adatas.obs["celltype"] = adatas.obs["clusters"].map(cluster2annotation).astype("category")
ordered_groups = list(marker_genes_dict.keys())
adatas.obs["celltype"] = (
    adatas.obs["celltype"]
    .astype("category")
    .cat.set_categories(ordered_groups, ordered=True)
)


#固定样本顺序
ordered_samples = sampleInfo
adatas.obs["sample"] = (
    adatas.obs["sample"]
    .astype("category")
    .cat.set_categories(ordered_samples, ordered=True)
)

ordered_groups = ['LT','LZ']
adatas.obs["group"] = (
    adatas.obs["group"]
    .astype("category")
    .cat.set_categories(ordered_groups, ordered=True)
)

In [ ]:
celltype_colors = {
    "SFT_Tumor": "#4C5F8F",
    "Pericyte/SMC": "#B7794B",
    "T/NK": "#6FA083",
    "Myeloid": "#B95F62",
    "Endo": "#A66C9A",
    "Mast": "#5BA8A0",
    "CAFs": "#8DA6BF",
    "Oligo": "#D8B56D",
    "OPCs": "#9BAE6A",
}

cats = adatas.obs["celltype"].cat.categories
adatas.uns["celltype_colors"] = [celltype_colors[x] for x in cats]

In [ ]:
group_colors = {
    "LT": "#4B6F83",
    "LZ": "#C58A54",
}

sample_colors = {
    "LT-a": "#4C5F8F",
    "LT-b": "#6D8BA6",
    "LT-c": "#6FA083",
    "LT-d": "#8E78B8",
    "LT-e": "#B95F62",
    "LT-f": "#B7794B",
    "LZ-c": "#C58A54",
    "LZ-d": "#D8B56D",
    "LZ-f": "#9BAE6A",
}

cats = adatas.obs["group"].cat.categories
adatas.uns["group_colors"] = [group_colors[x] for x in cats]

cats = adatas.obs["sample"].cat.categories
adatas.uns["sample_colors"] = [sample_colors[x] for x in cats]

In [ ]:
adatas.write_h5ad('./h5ad/rna_0.h5ad')